# 2. Supervised Modeling

En esta sección integramos el preprocesamiento modular del proyecto con varios clasificadores base de Scikit-Learn y evaluamos su desempeño con validación cruzada estratificada, priorizando Recall y F1 por el fuerte desbalance del dataset.

In [ ]:
# ==========================================
# 1. IMPORTS Y CONFIGURACION
# ==========================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn import set_config

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    parent_root = PROJECT_ROOT.parent
    if (parent_root / 'src').exists():
        PROJECT_ROOT = parent_root
    else:
        candidate = PROJECT_ROOT / 'mi_proyecto'
        if (candidate / 'src').exists():
            PROJECT_ROOT = candidate

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import UnknownToNaN, SmartImputer, OutlierCapper
from src.model_training import get_model_registry
from src.model_evaluation import build_stratified_kfold, print_model_comparison_report

set_config(display='diagram')

print(f'Proyecto detectado en: {PROJECT_ROOT}')
print('Imports sincronizados con src/ correctamente')

In [ ]:
# ==========================================
# 2. CARGA DE DATOS
# ==========================================

data_path = PROJECT_ROOT / 'data' / 'raw' / 'healthcare-dataset-stroke-data.csv'
df_raw = pd.read_csv(data_path)

print(f'Dataset cargado desde: {data_path}')
print(f'Dimensiones: {df_raw.shape[0]} filas, {df_raw.shape[1]} columnas')
df_raw.head()

In [ ]:
    # ==========================================
    # 3. DEFINICION DE VARIABLES
    # ==========================================

    target = 'stroke'
    X_raw = df_raw.drop(columns=[target, 'id'])
    y_raw = df_raw[target]

    numeric_features = X_raw.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()
    categorical_features = X_raw.select_dtypes(include=['object', 'string', 'category', 'bool']).columns.tolist()

    print(f'Features numéricas ({len(numeric_features)}): {numeric_features}')
    print(f'Features categóricas ({len(categorical_features)}): {categorical_features}')
    print('
Distribución del target:')
    print(y_raw.value_counts(normalize=True).mul(100).round(2).to_string())

In [ ]:
# ==========================================
# 4. PREPROCESAMIENTO BASE PARA LOS MODELOS
# ==========================================

feature_preprocessor = Pipeline([
    ('unknown_to_nan', UnknownToNaN(columns=categorical_features)),
    ('smart_imputer', SmartImputer()),
    ('outlier_capper', OutlierCapper(columns=numeric_features)),
    ('feature_encoding', ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('scaler', StandardScaler())
            ]), numeric_features),
            ('cat', Pipeline([
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ]), categorical_features),
        ],
        remainder='drop'
    ))
])

print('Preprocesador supervisado configurado con UnknownToNaN, SmartImputer, OutlierCapper, StandardScaler y OneHotEncoder')

In [ ]:
# ==========================================
# 5. PIPELINES DE MODELO + PREPROCESAMIENTO
# ==========================================

model_registry = get_model_registry(random_state=42)

def build_model_pipeline(estimator):
    """Construye un Pipeline completo con preprocesamiento y clasificador."""
    return Pipeline([
        ('preprocessing', feature_preprocessor),
        ('classifier', estimator),
    ])

model_pipelines = {
    name: build_model_pipeline(estimator)
    for name, estimator in model_registry.items()
}

print('Modelos base disponibles:')
for name in model_pipelines:
    print(f'- {name}')

In [ ]:
# ==========================================
# 7. RESUMEN FINAL
# ==========================================

best_model_name = prioritized_report.iloc[0]['model']
best_metrics = prioritized_report.iloc[0][['model', 'recall_mean', 'f1_mean', 'precision_mean', 'roc_auc_mean']]

print(f'\nMejor candidato según Recall/F1: {best_model_name}')
print(best_metrics.to_string())

prioritized_report.head(3)


In [ ]:
    # ==========================================
    # 7. REPORTE DETALLADO DEL MEJOR CANDIDATO
    # ==========================================

    best_model_name = prioritized_report.iloc[0]['model']
    best_model_pipeline = model_pipelines[best_model_name]

    print(f'
Mejor candidato según Recall/F1: {best_model_name}')
    best_fold_report = print_classification_cv_report(
        model_name=best_model_name,
        estimator=best_model_pipeline,
        X=X_raw,
        y=y_raw,
        cv=cv,
        random_state=42,
    )
    best_fold_report